# Reproduction et Évaluation Complète du Modèle IA de Recouvrement (BWAI 2026 — Cas 1)

Ce notebook permet de **reproduire intégralement** l'entraînement, l'évaluation *hold-out*, la comparaison à une méthode de référence (*baseline* par mots-clés), la **courbe de calibration du seuil Couverture / Erreur** et l'audit des **garde-fous critiques** (`services/ai_classifier.py`).

## Objectifs du notebook (Couverture 100 % de la Partie B)
1. **Charger et auditer** `data/train.csv` (250 messages) et `data/test.csv` (100 messages *hold-out*), et vérifier l'absence de fuite de données (*data leakage*).
2. **Reproduire l'entraînement** du pipeline `TF-IDF (1-2 grammes, normalisation Unicode) + LogisticRegression (C=2.0, class_weight='balanced', random_state=42)`.
3. **Mesurer les performances** sur `data/test.csv` : Accuracy, Précision, Rappel, F1-score par classe, Matrice de confusion, et performance sur les cas ambigus (`ambigu=oui`).
4. **Comparer à une baseline déterministe** par mots-clés pour quantifier le gain réel de l'approche IA.
5. **Calibrer mathématiquement le seuil de confiance** via la courbe Couverture / Erreur (`0.35` à `0.75`) et tester le **garde-fou lexical critique** (détection de négation ou de montant promis sur `RETARD_SIMPLE`).

## 1. Chargement des données et contrôle d'étanchéité (Train vs Hold-out Test)

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

TRAIN_PATH = PROJECT_ROOT / "data" / "train.csv"
TEST_PATH = PROJECT_ROOT / "data" / "test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Jeu d'entraînement ({TRAIN_PATH.name}) : {train_df.shape[0]} lignes, colonnes = {list(train_df.columns)}")
print(f"Jeu de test hold-out ({TEST_PATH.name}) : {test_df.shape[0]} lignes, colonnes = {list(test_df.columns)}")

train_set = set(train_df["message"].str.strip().str.lower())
test_set = set(test_df["message"].str.strip().str.lower())
overlap = train_set & test_set

print(f"Valeurs manquantes (Train / Test) : {train_df.isnull().sum().sum()} / {test_df.isnull().sum().sum()}")
print(f"Doublons internes (Train / Test)  : {len(train_df) - len(train_set)} / {len(test_df) - len(test_set)}")
print(f"Chevauchement exact Train ∩ Test  : {len(overlap)} message(s) en commun")
assert len(overlap) == 0, "Erreur : fuite de données entre train.csv et test.csv !"

Jeu d'entraînement (train.csv) : 250 lignes, colonnes = ['id', 'message', 'classe', 'ambigu']
Jeu de test hold-out (test.csv) : 100 lignes, colonnes = ['id', 'message', 'classe', 'ambigu']
Valeurs manquantes (Train / Test) : 0 / 0
Doublons internes (Train / Test)  : 0 / 0
Chevauchement exact Train ∩ Test  : 0 message(s) en commun


### Répartition des classes et proportion de messages ambigus

In [2]:
summary_df = pd.DataFrame({
    "Train (n=250)": train_df["classe"].value_counts(),
    "Train (%)": (train_df["classe"].value_counts(normalize=True) * 100).round(1),
    "Test Hold-out (n=100)": test_df["classe"].value_counts(),
    "Test (%)": (test_df["classe"].value_counts(normalize=True) * 100).round(1),
})
print("=== Répartition par classe ===")
print(summary_df.to_string())

print("\n=== Répartition de l'attribut 'ambigu' ===")
ambigu_df = pd.DataFrame({
    "Train": train_df["ambigu"].value_counts(),
    "Test Hold-out": test_df["ambigu"].value_counts(),
})
print(ambigu_df.to_string())

=== Répartition par classe ===
                    Train (n=250)  Train (%)  Test Hold-out (n=100)  Test (%)
classe                                                                       
ANNONCE_PAIEMENT               50       20.0                     32      32.0
DEMANDE_ECHEANCIER             50       20.0                     26      26.0
LITIGE                        100       40.0                     24      24.0
RETARD_SIMPLE                  50       20.0                     18      18.0

=== Répartition de l'attribut 'ambigu' ===
        Train  Test Hold-out
ambigu                      
non       221             88
oui        29             12


## 2. Entraînement reproductible du modèle IA (`TF-IDF + LogisticRegression`)

In [3]:
LABELS_ORDER = ["LITIGE", "DEMANDE_ECHEANCIER", "ANNONCE_PAIEMENT", "RETARD_SIMPLE"]

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), lowercase=True, strip_accents="unicode")),
    ("clf", LogisticRegression(C=2.0, class_weight="balanced", random_state=42, max_iter=1000)),
])

pipeline.fit(train_df["message"], train_df["classe"])
vocab_size = len(pipeline.named_steps["tfidf"].vocabulary_)
print(f"Modèle entraîné avec succès sur {len(train_df)} messages.")
print(f"Taille du vocabulaire TF-IDF (1-2 grammes) : {vocab_size} caractéristiques.")

Modèle entraîné avec succès sur 250 messages.
Taille du vocabulaire TF-IDF (1-2 grammes) : 2618 caractéristiques.


## 3. Évaluation quantitative sur `data/test.csv` et comparaison à la baseline mots-clés

In [4]:
from services.ai_classifier import _keyword_baseline_predict

probs = pipeline.predict_proba(test_df["message"])
classes = list(pipeline.classes_)
y_pred = [classes[i] for i in probs.argmax(axis=1)]
y_conf = [round(float(p.max()), 3) for p in probs]
y_true = list(test_df["classe"])

baseline_preds = [_keyword_baseline_predict(m) for m in test_df["message"]]

ai_acc = accuracy_score(y_true, y_pred)
base_acc = accuracy_score(y_true, baseline_preds)

print(f"Accuracy Modèle IA (TF-IDF + LogReg) : {ai_acc:.2%}")
print(f"Accuracy Baseline par mots-clés      : {base_acc:.2%}")
print(f"Gain net du modèle IA vs Baseline    : +{(ai_acc - base_acc) * 100:.2f} points\n")

print("=== Rapport de classification détaillé (Modèle IA sur test.csv) ===")
print(classification_report(y_true, y_pred, labels=LABELS_ORDER, digits=4))

Accuracy Modèle IA (TF-IDF + LogReg) : 89.00%
Accuracy Baseline par mots-clés      : 36.00%
Gain net du modèle IA vs Baseline    : +53.00 points

=== Rapport de classification détaillé (Modèle IA sur test.csv) ===
                    precision    recall  f1-score   support

            LITIGE     0.7667    0.9583    0.8519        24
DEMANDE_ECHEANCIER     0.9231    0.9231    0.9231        26
  ANNONCE_PAIEMENT     0.9259    0.7812    0.8475        32
     RETARD_SIMPLE     1.0000    0.9444    0.9714        18

          accuracy                         0.8900       100
         macro avg     0.9039    0.9018    0.8985       100
      weighted avg     0.9003    0.8900    0.8905       100


### Matrice de confusion et analyse par niveau d'ambiguïté (`ambigu=non` vs `ambigu=oui`)

In [5]:
cm = confusion_matrix(y_true, y_pred, labels=LABELS_ORDER)
cm_df = pd.DataFrame(
    cm,
    index=[f"Vrai : {l}" for l in LABELS_ORDER],
    columns=[f"Prédit : {l}" for l in LABELS_ORDER],
)
print("=== Matrice de confusion (100 messages hold-out) ===")
print(cm_df.to_string())

eval_df = test_df.copy()
eval_df["pred"] = y_pred
eval_df["confiance"] = y_conf
eval_df["correct"] = eval_df["classe"] == eval_df["pred"]

std_sub = eval_df[eval_df["ambigu"] == "non"]
amb_sub = eval_df[eval_df["ambigu"] == "oui"]

print(f"\nAccuracy sur messages standards (ambigu=non, n={len(std_sub)}) : {std_sub['correct'].mean():.2%} ({std_sub['correct'].sum()}/{len(std_sub)})")
print(f"Accuracy sur messages ambigus   (ambigu=oui, n={len(amb_sub)}) : {amb_sub['correct'].mean():.2%} ({amb_sub['correct'].sum()}/{len(amb_sub)})")

=== Matrice de confusion (100 messages hold-out) ===
                           Prédit : LITIGE  Prédit : DEMANDE_ECHEANCIER  Prédit : ANNONCE_PAIEMENT  Prédit : RETARD_SIMPLE
Vrai : LITIGE                           23                            1                          0                       0
Vrai : DEMANDE_ECHEANCIER                1                           24                          1                       0
Vrai : ANNONCE_PAIEMENT                  6                            1                         25                       0
Vrai : RETARD_SIMPLE                     0                            0                          1                      17

Accuracy sur messages standards (ambigu=non, n=88) : 88.64% (78/88)
Accuracy sur messages ambigus   (ambigu=oui, n=12) : 91.67% (11/12)


## 4. Calibration du seuil (Courbe Couverture / Erreur) et Garde-fou lexical critique (Points B.3 & B.4)

Plutôt que d'imposer un seuil de confiance sans justification, nous balayons les seuils $\tau \in [0.35, 0.75]$ via `classifier_singleton.calibrate_thresholds()` et vérifions également le garde-fou lexical `detect_critical_patterns` (qui force la revue humaine si un message prédit `RETARD_SIMPLE` contient une négation ou un montant chiffré) :

In [6]:
from services.ai_classifier import classifier_singleton, detect_critical_patterns

curve = classifier_singleton.calibrate_thresholds()
curve_df = pd.DataFrame(curve)
print("=== Courbe de calibration Couverture / Erreur selon le seuil de confiance ===")
print(curve_df.to_string(index=False))

print("\n=== Démonstration du garde-fou lexical critique sur RETARD_SIMPLE ===")
sample_critical_msgs = [
    "Desole pour le retard, je ne refuse pas de payer, je regle rapidement",
    "Desole pour cet oubli sur les 25000 fcfa, je m'en occupe rapidement",
]
for m in sample_critical_msgs:
    res = classifier_singleton.classify(m, confidence_threshold=0.20)
    print(f"Message : {m}")
    print(f" -> Catégorie={res['categorie']}, Confiance={res['confiance']}, Garde-fou={res['garde_fou_critique']}, Revue={res['requiert_revue_humaine']}")
    print(f" -> Motif : {res['motif_revue']}\n")

=== Courbe de calibration Couverture / Erreur selon le seuil de confiance ===
 threshold  ai_coverage  ai_precision_above_threshold  ai_errors_above_threshold  ai_errors_below_threshold  total_ai_errors  business_auto_rate  business_human_review_rate  business_unintercepted_errors  critical_false_negatives
      0.35         0.90                        0.9111                          8                          3               11                0.13                        0.87                              0                         0
      0.40         0.80                        0.9125                          7                          4               11                0.12                        0.88                              0                         0
      0.45         0.60                        0.9833                          1                         10               11                0.12                        0.88                              0                         0
  

## 5. Inspection complète des 11 erreurs observées sur `data/test.csv`

In [7]:
prod_report = classifier_singleton.evaluate_holdout()
for err in prod_report["errors_detail"]:
    print(
        f"[{err['id']}] Vrai={err['vrai_label']:18s} | Prédit={err['pred_label']:18s} "
        f"| Conf={err['confiance']:.3f} | Ambigu={err['ambigu']:3s} | Intercepté_Revue={err['intercepte_revue_humaine']}"
    )
    print(f"   Message : {err['message']}\n")

[T010] Vrai=LITIGE             | Prédit=DEMANDE_ECHEANCIER | Conf=0.349 | Ambigu=non | Intercepté_Revue=True
   Message : Les 15 sacs de ciment livrés sur le chantier sont en partie durcis. Je paierai après inspection par un expert

[T036] Vrai=DEMANDE_ECHEANCIER | Prédit=LITIGE             | Conf=0.440 | Ambigu=non | Intercepté_Revue=True
   Message : Le loyer de la boutique est tombé en même temps que la facture. Je demande un étalement de la facture sur deux mois

[T050] Vrai=DEMANDE_ECHEANCIER | Prédit=ANNONCE_PAIEMENT   | Conf=0.438 | Ambigu=non | Intercepté_Revue=True
   Message : J'aimerais payer en trois tranches égales, je vous envoie la première dès demain si vous confirmez

[T055] Vrai=ANNONCE_PAIEMENT   | Prédit=LITIGE             | Conf=0.401 | Ambigu=non | Intercepté_Revue=True
   Message : Ne vous inquiétez pas, je passe au cabinet demain avant midi avec le chèque

[T059] Vrai=ANNONCE_PAIEMENT   | Prédit=DEMANDE_ECHEANCIER | Conf=0.367 | Ambigu=non | Intercepté_Revue=Tru

## 6. Bilan final de conformité avec `services/ai_classifier.py`

In [8]:
assert prod_report["accuracy"] == round(ai_acc, 4)
assert prod_report["critical_false_negatives_auto_sent"] == 0
assert prod_report["errors_caught_by_human_review"] == prod_report["total_errors"]

print("Bilan de conformité 100 % validé :")
print({
    "train_size": prod_report["train_size"],
    "test_size": prod_report["test_size"],
    "accuracy": prod_report["accuracy"],
    "macro_f1": prod_report["macro_f1"],
    "baseline_keyword_accuracy": prod_report["baseline_keyword_accuracy"],
    "errors_caught_by_human_review": f"{prod_report['errors_caught_by_human_review']}/{prod_report['total_errors']}",
    "critical_false_negatives_auto_sent": prod_report["critical_false_negatives_auto_sent"],
})

Bilan de conformité 100 % validé :
{'train_size': 250, 'test_size': 100, 'accuracy': 0.89, 'macro_f1': 0.8985, 'baseline_keyword_accuracy': 0.36, 'errors_caught_by_human_review': '11/11', 'critical_false_negatives_auto_sent': 0}
